# VAYU — Data Audit
Run this notebook first. It will scan your entire /data folder and tell us exactly what we're working with.

In [ ]:
# ── Install if needed ──────────────────────────────
# !pip install pandas openpyxl chardet tqdm -q

In [ ]:
import os, glob, warnings, chardet
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

# ── CHANGE THIS to your data folder path ──────────
DATA_ROOT = './data'
# ──────────────────────────────────────────────────

csv_files  = glob.glob(os.path.join(DATA_ROOT, '**/*.csv'),  recursive=True)
xlsx_files = glob.glob(os.path.join(DATA_ROOT, '**/*.xlsx'), recursive=True)
xls_files  = glob.glob(os.path.join(DATA_ROOT, '**/*.xls'),  recursive=True)
all_files  = csv_files + xlsx_files + xls_files

print(f'📁 CSV  files : {len(csv_files)}')
print(f'📊 XLSX files : {len(xlsx_files)}')
print(f'📊 XLS  files : {len(xls_files)}')
print(f'📦 TOTAL      : {len(all_files)}')
print()

# Show folder structure (2 levels deep)
print('── Folder structure ──────────────────────')
seen = set()
for f in all_files:
    parts = Path(f).relative_to(DATA_ROOT).parts
    for depth in range(1, min(3, len(parts))):
        folder = os.path.join(*parts[:depth])
        if folder not in seen:
            print('  ' * (depth-1) + '📂 ' + parts[depth-1])
            seen.add(folder)
print('──────────────────────────────────────────')

In [ ]:
# ── Profile every file ─────────────────────────────

def detect_encoding(fp):
    with open(fp, 'rb') as f:
        return chardet.detect(f.read(30000)).get('encoding', 'utf-8') or 'utf-8'

def read_any(fp):
    ext = Path(fp).suffix.lower()
    if ext in ['.xlsx', '.xls']:
        xl = pd.ExcelFile(fp)
        dfs = [pd.read_excel(fp, sheet_name=s) for s in xl.sheet_names]
        return max(dfs, key=len) if dfs else None
    enc = detect_encoding(fp)
    for sep in [',', ';', '\t', '|']:
        try:
            df = pd.read_csv(fp, sep=sep, encoding=enc, errors='replace',
                             low_memory=False, nrows=500)
            if df.shape[1] > 1:
                return df
        except Exception:
            continue
    return None

records = []
POLLUTANT_KW = ['pm2', 'pm10', 'no2', 'so2', 'co', 'o3', 'aqi',
                'nh3', 'nox', 'benzene', 'pollutant']
DATE_KW      = ['date', 'time', 'timestamp', 'from', 'to']
LOCATION_KW  = ['station', 'city', 'state', 'location', 'site', 'area']
JUNK_KW      = ['weather', 'rain', 'wind', 'temperature', 'humidity',
                'traffic', 'noise', 'population', 'hospital', 'crime']

for fp in tqdm(all_files, desc='Auditing files'):
    rec = {
        'file'           : os.path.basename(fp),
        'relative_path'  : str(Path(fp).relative_to(DATA_ROOT)),
        'extension'      : Path(fp).suffix.lower(),
        'size_kb'        : round(os.path.getsize(fp) / 1024, 1),
        'rows'           : None,
        'cols'           : None,
        'columns'        : '',
        'has_pollutants' : False,
        'has_dates'      : False,
        'has_location'   : False,
        'looks_like_junk': False,
        'date_sample'    : '',
        'missing_pct'    : '',
        'sentinel_found' : '',
        'relevance'      : 'UNKNOWN',
        'error'          : '',
    }
    try:
        df = read_any(fp)
        if df is None or df.empty:
            rec['error'] = 'empty or unreadable'
            records.append(rec)
            continue

        cols_lower = [c.lower().strip() for c in df.columns]
        rec['rows']    = len(df)
        rec['cols']    = len(df.columns)
        rec['columns'] = ' | '.join(df.columns.tolist())

        rec['has_pollutants'] = any(any(kw in c for kw in POLLUTANT_KW) for c in cols_lower)
        rec['has_dates']      = any(any(kw in c for kw in DATE_KW)      for c in cols_lower)
        rec['has_location']   = any(any(kw in c for kw in LOCATION_KW)  for c in cols_lower)
        rec['looks_like_junk']= (not rec['has_pollutants'] and
                                  any(any(kw in c for kw in JUNK_KW) for c in cols_lower))

        # Sample date column value
        for col in df.columns:
            if any(kw in col.lower() for kw in DATE_KW):
                rec['date_sample'] = str(df[col].dropna().iloc[0]) if df[col].notna().any() else ''
                break

        # Missing % overall
        rec['missing_pct'] = f"{df.isnull().mean().mean()*100:.1f}%"

        # Sentinel check
        sentinels_found = []
        for sv in [-999, -9999, 999, 9999]:
            if (df == sv).any().any():
                sentinels_found.append(str(sv))
        rec['sentinel_found'] = ', '.join(sentinels_found)

        # Auto-tag relevance
        if rec['has_pollutants'] and rec['has_dates']:
            rec['relevance'] = '✅ RELEVANT'
        elif rec['has_pollutants'] and not rec['has_dates']:
            rec['relevance'] = '⚠️  PARTIAL (no date)'
        elif rec['looks_like_junk']:
            rec['relevance'] = '❌ LIKELY IRRELEVANT'
        else:
            rec['relevance'] = '❓ NEEDS REVIEW'

    except Exception as e:
        rec['error'] = str(e)

    records.append(rec)

audit_df = pd.DataFrame(records)
print('\n✅ Audit complete!')

In [ ]:
# ── Summary ────────────────────────────────────────
print('='*55)
print('  RELEVANCE SUMMARY')
print('='*55)
print(audit_df['relevance'].value_counts().to_string())
print()
print('='*55)
print('  FILES WITH ERRORS')
print('='*55)
err = audit_df[audit_df['error'] != '']
print(f'{len(err)} files could not be read')
if len(err): print(err[['file','error']].to_string())
print()
print('='*55)
print('  SENTINEL VALUES FOUND IN')
print('='*55)
sent = audit_df[audit_df['sentinel_found'] != '']
print(f'{len(sent)} files have -999/999 type sentinels')

In [ ]:
# ── Show RELEVANT files ────────────────────────────
relevant = audit_df[audit_df['relevance'] == '✅ RELEVANT']
print(f'RELEVANT files: {len(relevant)}')
display(relevant[['file','relative_path','rows','cols','columns','missing_pct','date_sample','sentinel_found']].head(30))

In [ ]:
# ── Show NEEDS REVIEW files ────────────────────────
review = audit_df[audit_df['relevance'] == '❓ NEEDS REVIEW']
print(f'NEEDS REVIEW: {len(review)}')
display(review[['file','relative_path','rows','cols','columns']].head(30))

In [ ]:
# ── Show IRRELEVANT files ──────────────────────────
junk = audit_df[audit_df['relevance'] == '❌ LIKELY IRRELEVANT']
print(f'LIKELY IRRELEVANT: {len(junk)}')
display(junk[['file','relative_path','columns']].head(30))

In [ ]:
# ── Unique column names across ALL files ───────────
# This reveals schema inconsistencies
all_cols = []
for cols in audit_df['columns']:
    all_cols.extend([c.strip().lower() for c in cols.split('|') if c.strip()])

col_counts = pd.Series(all_cols).value_counts()
print('Top 40 column names across all files:')
print(col_counts.head(40).to_string())

In [ ]:
# ── Date format samples ────────────────────────────
print('Unique date format samples found:')
print(audit_df['date_sample'].dropna().unique()[:30])

In [ ]:
# ── Save audit to CSV ──────────────────────────────
audit_df.to_csv('vayu_audit_report.csv', index=False)
print('Saved → vayu_audit_report.csv')
print('\nShare this file / paste the output of cells above')
print('so we can proceed with the cleaning notebook.')